# Loading your BWK sample points.
## Loading packages
Let's start with installing the required packages.

In [21]:
%pip install "geotessera==0.8.0"

In [22]:
try:
    import ee, geemap
except ImportError:
    !pip -q install earthengine-api geemap

import ee, geemap, datetime # ee: Google Earth Engine (GEE) API, geemap: mapping GEE products
import pandas as pd
import matplotlib.pyplot as plt # for figure plotting
import geopandas as gpd # for geospatial dataframes
import os

In [23]:
%pip install "leafmap[maplibre]" # For interactive visualization of maps.
from maplibre.plugins import MapboxDrawControls, MapboxDrawOptions # To select ROI's on your maps.

Connect to your Google Drive to load your dataset within your drive.

In [24]:
from google.colab import drive
drive.mount('/content/drive/')

Drive already mounted at /content/drive/; to attempt to forcibly remount, call drive.mount("/content/drive/", force_remount=True).


In [ ]:
import os
import geopandas as gpd
import numpy as np
import pandas as pd
from geotessera.store import GeoTesseraZarr

# 1. Base directories and file patterns
base_input_dir = "/content/drive/Shareddrives/PRJ_SlimmeBeeldverwerking/PRJ_2026_SlimmeBeeldverwerking/BWK - graslandfases/selecties"
output_dir = "/content/drive/Shareddrives/PRJ_SlimmeBeeldverwerking/PRJ_2026_SlimmeBeeldverwerking/Slimmebeeldverwerking/extract_embeddings/OutputTesseraEmbeddings"
os.makedirs(output_dir, exist_ok=True)

# 2. Initialize GeoTessera Zarr reader once
gt = GeoTesseraZarr()

# 3. Inventory years to process
inventory_years = [2019, 2020, 2021, 2022, 2023]

for inv_year in inventory_years:
  print(f"\n{'='*50}\nProcessing Inventory Year: {inv_year}\n{'='*50}")

  grass_path = os.path.join(
      base_input_dir, f"Grassland_classified_final_classes_{inv_year}.gpkg"
  )
  nograss_path = os.path.join(
      base_input_dir, f"NoGrassland_classified_final_classes_{inv_year}.gpkg"
  )

  if not os.path.exists(grass_path) or not os.path.exists(nograss_path):
    print(f"Skipping {inv_year}: files not found.")
    continue

  # Load datasets
  grasslands = gpd.read_file(grass_path)
  nograsslands = gpd.read_file(nograss_path)

  # Sample 30% of nograsslands
  samplesize = round(len(grasslands) * 30 / 100)
  nograsslands_samp = nograsslands.sample(
      n=min(samplesize, len(nograsslands)), random_state=42
  )
  nograsslands_samp = nograsslands_samp.assign(Level4="nograss")
  nograsslands_samp = nograsslands_samp.to_crs(grasslands.crs)

  # Combine and convert to centroids
  datagrassbwk = pd.concat([grasslands, nograsslands_samp], ignore_index=True)
  datagrassbwk["geometry"] = datagrassbwk.geometry.centroid

  # Convert to WGS84 (lon, lat) for GeoTessera
  points = datagrassbwk.to_crs("EPSG:4326")[["Level4", "geometry"]].copy()
  coords = [(geom.x, geom.y) for geom in points.geometry]

  # Metadata columns (Level4, lon, lat)
  base_out = pd.DataFrame({
      "Level4": points["Level4"].values,
      "lon": points.geometry.x.values,
      "lat": points.geometry.y.values,
      "inventory_year": inv_year,
  })

  # Embeddings for: year - 2, year - 1, and year
  emb_years = [inv_year - 2, inv_year - 1, inv_year]
  year_dfs = []

  for emb_year in emb_years:
    print(
        f"Sampling {len(coords)} points for embedding year {emb_year} (inventory"
        f" {inv_year})..."
    )
    X = gt.sample_points(coords, year=emb_year)

    # Prefix columns with year to avoid collisions (e.g., emb_2017_0, ..., emb_2017_127)
    cols = [f"emb_{emb_year}_{i}" for i in range(X.shape[1])]
    year_dfs.append(pd.DataFrame(X, columns=cols))

  # Combine metadata and multi-year embeddings into a single DataFrame
  out_final = pd.concat([base_out] + year_dfs, axis=1)

  # Save to CSV
  out_csv = os.path.join(output_dir, f"Tessera_inventory_{inv_year}_3yr.csv")
  out_final.to_csv(out_csv, index=False)
  print(f"Saved: {out_csv} | Shape: {out_final.shape}")